# Customer Segmentation Analysis

## Project Overview
Analyze customer behavior data to segment customers using descriptive statistics, RFM analysis, and exploratory clustering-style visualization. This is a **data analysis** project — the focus is on understanding customer groups through EDA, not on building ML models.

## Learning Objectives
- Build RFM (Recency, Frequency, Monetary) features from transaction data
- Segment customers using quantile-based grouping
- Visualize segment characteristics with rich charts
- Derive actionable marketing insights from segments

## Problem Statement
An e-commerce business wants to understand its customer base. By segmenting customers based on purchasing behavior, the business can tailor marketing, retention, and upselling strategies to each group.

## Why This Project Matters
Customer segmentation is the foundation of personalized marketing. Understanding which customers are high-value, at-risk, or newly acquired lets businesses allocate resources efficiently and improve lifetime value.

## Dataset Overview
We use the **Online Retail II** dataset (UCI ML Repository). It contains ~500K invoice-level transactions from a UK-based online retailer (2009-2011). Key columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.

## Dataset Source and License Notes
- **Source**: UCI Machine Learning Repository — Online Retail II
- **License**: CC BY 4.0
- **URL**: https://archive.ics.uci.edu/dataset/502/online+retail+ii
- The dataset is freely available for educational and research purposes.

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 20)
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
MAX_ROWS = 200_000  # cap for speed
SNAPSHOT_DATE_OFFSET = 1  # days after last invoice for recency calc
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading
We generate realistic synthetic data that mirrors the Online Retail dataset structure. This keeps the notebook fully self-contained with no external downloads required.

In [ ]:
np.random.seed(42)
n_customers = 2000
n_transactions = 50000

customer_ids = np.random.randint(10000, 19999, size=n_transactions)
# Ensure only n_customers unique customers
unique_pool = np.random.choice(np.arange(10000, 10000 + n_customers), size=n_transactions, replace=True)
customer_ids = unique_pool

dates = pd.date_range('2023-01-01', '2024-12-31', freq='h')
invoice_dates = np.random.choice(dates, size=n_transactions)

quantities = np.random.randint(1, 20, size=n_transactions)
unit_prices = np.round(np.random.lognormal(mean=1.5, sigma=0.8, size=n_transactions), 2)
unit_prices = np.clip(unit_prices, 0.5, 200)

countries = np.random.choice(
    ['United Kingdom', 'Germany', 'France', 'Spain', 'Netherlands', 'Australia'],
    size=n_transactions, p=[0.60, 0.10, 0.10, 0.08, 0.07, 0.05]
)

df = pd.DataFrame({
    'InvoiceNo': [f'INV{i:06d}' for i in range(n_transactions)],
    'CustomerID': customer_ids,
    'InvoiceDate': invoice_dates,
    'Quantity': quantities,
    'UnitPrice': unit_prices,
    'Country': countries,
})
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df = df.sort_values('InvoiceDate').reset_index(drop=True)
print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Unique customers: {df["CustomerID"].nunique()}')
print(f'Date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')
print(f'Negative quantities: {(df["Quantity"] < 0).sum()}')
print(f'Zero/negative prices: {(df["UnitPrice"] <= 0).sum()}')

# Remove any bad rows
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
print(f'\nClean shape: {df.shape}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue distribution
axes[0,0].hist(df['TotalAmount'], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Distribution of Transaction Amounts')
axes[0,0].set_xlabel('Total Amount')

# Transactions per country
country_counts = df['Country'].value_counts()
country_counts.plot.bar(ax=axes[0,1], edgecolor='black')
axes[0,1].set_title('Transactions by Country')
axes[0,1].tick_params(axis='x', rotation=45)

# Monthly transaction volume
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('YearMonth').size()
monthly.plot(ax=axes[1,0], marker='o')
axes[1,0].set_title('Monthly Transaction Volume')
axes[1,0].tick_params(axis='x', rotation=45)

# Revenue per month
monthly_rev = df.groupby('YearMonth')['TotalAmount'].sum()
monthly_rev.plot(ax=axes[1,1], marker='s', color='coral')
axes[1,1].set_title('Monthly Revenue')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()
print('EDA overview saved.')

## Target Analysis
In a segmentation project, there is no single prediction target. Instead, we analyze the distribution of customer-level behavioral features that will define segments.

In [ ]:
customer_stats = df.groupby('CustomerID').agg(
    total_spend=('TotalAmount', 'sum'),
    num_transactions=('InvoiceNo', 'count'),
    avg_order_value=('TotalAmount', 'mean'),
    num_unique_dates=('InvoiceDate', lambda x: x.dt.date.nunique()),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in zip(axes,
    ['total_spend', 'num_transactions', 'avg_order_value'],
    ['Total Spend', 'Transaction Count', 'Avg Order Value']):
    ax.hist(customer_stats[col], bins=40, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution of {title}')
    ax.set_xlabel(title)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'customer_distributions.png'), dpi=100, bbox_inches='tight')
plt.show()
print(customer_stats.describe().round(2))

## RFM Feature Engineering
RFM stands for **Recency** (days since last purchase), **Frequency** (number of purchases), and **Monetary** (total spend). These three features capture the essential purchasing behavior of each customer.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + timedelta(days=SNAPSHOT_DATE_OFFSET)
print(f'Snapshot date: {snapshot_date}')

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'count'),
    Monetary=('TotalAmount', 'sum'),
).reset_index()

print(f'RFM table shape: {rfm.shape}')
rfm.describe().round(2)

## Quantile-Based Segmentation
We assign each customer an R, F, and M score (1–4) based on quartiles. Lower recency is better (scored 4), while higher frequency and monetary are better (scored 4).

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print(rfm[['CustomerID', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head(10))
print(f'\nRFM Score distribution:\n{rfm["RFM_Score"].value_counts().sort_index()}')

## Segment Labeling
We map RFM composite scores to business-friendly segment names.

In [ ]:
def label_segment(score):
    if score >= 10:
        return 'Champions'
    elif score >= 8:
        return 'Loyal Customers'
    elif score >= 6:
        return 'Potential Loyalists'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost / Low Value'

rfm['Segment'] = rfm['RFM_Score'].apply(label_segment)
segment_counts = rfm['Segment'].value_counts()
print(segment_counts)

## Segment Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Segment distribution
segment_counts.plot.bar(ax=axes[0], edgecolor='black', color=sns.color_palette('Set2'))
axes[0].set_title('Customer Count by Segment')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Revenue by segment
seg_revenue = rfm.groupby('Segment')['Monetary'].sum().reindex(segment_counts.index)
seg_revenue.plot.bar(ax=axes[1], edgecolor='black', color=sns.color_palette('Set3'))
axes[1].set_title('Total Revenue by Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Revenue')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'segments.png'), dpi=100, bbox_inches='tight')
plt.show()

## Segment Profile Deep Dive

In [ ]:
segment_profile = rfm.groupby('Segment').agg(
    count=('CustomerID', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
    median_monetary=('Monetary', 'median'),
    total_revenue=('Monetary', 'sum'),
).round(2)
segment_profile['pct_customers'] = (segment_profile['count'] / segment_profile['count'].sum() * 100).round(1)
segment_profile['pct_revenue'] = (segment_profile['total_revenue'] / segment_profile['total_revenue'].sum() * 100).round(1)
print(segment_profile)

## RFM Heatmap

In [ ]:
heatmap_data = rfm.groupby(['R_Score', 'F_Score']).size().unstack(fill_value=0)
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Customer Count: Recency vs Frequency Scores')
plt.xlabel('Frequency Score')
plt.ylabel('Recency Score')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'rfm_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Spend vs Frequency by Segment

In [ ]:
plt.figure(figsize=(10, 7))
for seg in rfm['Segment'].unique():
    mask = rfm['Segment'] == seg
    plt.scatter(rfm.loc[mask, 'Frequency'], rfm.loc[mask, 'Monetary'],
                label=seg, alpha=0.6, s=30)
plt.xlabel('Frequency (# transactions)')
plt.ylabel('Monetary (total spend)')
plt.title('Spend vs Frequency by Segment')
plt.legend(title='Segment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'spend_vs_freq.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Champions** (high R, F, M): Most valuable customers — reward and retain them with VIP programs.
- **Loyal Customers**: Consistent buyers — upsell new product lines.
- **Potential Loyalists**: Moderate engagement — nurture with targeted campaigns to move them up.
- **At Risk**: Declining recency or frequency — trigger re-engagement campaigns.
- **Lost / Low Value**: Inactive or minimal spend — decide between win-back offers or resource reallocation.

The Pareto principle often holds: a small percentage of customers drive a disproportionate share of revenue.

## Limitations
- Synthetic data used here may not capture real-world skewness (e.g., power-law spending distributions)
- RFM segmentation is rule-based and does not account for product preferences or demographics
- Quartile boundaries are arbitrary; domain experts should validate segment cutoffs
- No temporal validation — segment labels may shift over different time windows

## How to Improve This Project
- Use real transaction data for more realistic distributions
- Apply K-Means or HDBSCAN clustering on RFM features for data-driven segmentation
- Add CLV (Customer Lifetime Value) predictions per segment
- Incorporate demographic or behavioral features beyond RFM
- Build a time-series view: track how customers migrate between segments over time

## Production Considerations
- Automate RFM scoring on a monthly or weekly cadence
- Store segment labels in a CRM system for campaign targeting
- Monitor segment drift: if Champions shrink quarter-over-quarter, investigate causes
- Define SLAs for re-engagement trigger timing

## Common Mistakes
- Using total data (without time window) makes Recency meaningless
- Not removing returns/cancellations inflates Frequency and Monetary
- Treating RFM scores as continuous rather than ordinal
- Applying equal quartile splits to heavily skewed data without checking

## Mini Challenge / Exercises
1. Redefine segments using 5 quantile bins instead of 4. How does segment distribution change?
2. Filter to only UK customers and re-run segmentation. Do the same patterns hold?
3. Add a 4th dimension: average basket size. How would you label a 4D segmentation?
4. Plot segment migration between H1 2023 and H2 2023.

## Final Summary / Key Takeaways
- RFM analysis is a simple yet powerful framework for understanding customer behavior
- Quantile-based scoring provides a fast, interpretable segmentation baseline
- Even without ML, clear business actions emerge from segment profiles
- Champions and Loyal Customers typically drive the majority of revenue
- Segment monitoring and re-engagement triggers are essential for retention